In [0]:
from pyspark.sql import SparkSession

In [0]:
data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)

### create delta table


In [0]:
# store data as delta format
df.write.format("delta").mode("overWrite").save("/Volumes/workspace/default/delta_as")
     

In [0]:
# load data
df1= spark.read.format("delta").load("/Volumes/workspace/default/delta_as")
display(df1)

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


### Task 2

In [0]:
#new data insertion
new_data=[(5,"C005" ,"Camera", 30000)]
columns=["id","customer_id","product","amount"]
new_df=spark.createDataFrame(new_data,columns)


In [0]:
new_df.write.format("delta").mode("append").save("/Volumes/workspace/default/delta_as")


In [0]:
existing_df=spark.read.load("/Volumes/workspace/default/delta_as")
display(existing_df)

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


### Update data

In [0]:
#id=2 -> amount=18000
from delta.tables import DeltaTable
from pyspark.sql.functions import lit
deltatable=DeltaTable.forPath(spark,"/Volumes/workspace/default/delta_as")
deltatable.update("id=2",set={"amount":lit(18000)})
deltatable.toDF().display()

id,customer_id,product,amount
1,C001,Laptop,50000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000
2,C002,Mobile,18000


### Task 4

In [0]:
# Delete data
deltatable.delete('id=1')
deltatable.toDF().display()

id,customer_id,product,amount
2,C002,Mobile,18000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


### Task 5

In [0]:
# MERGE (Incremental Load)

 #New incoming data:

#updates = [
  #  (3, "C003", "Tablet", 22000),  # update
   # (6, "C006", "Watch", 8000)     # insert
#]

target_df=spark.read.load("/Volumes/volume1/default/delta_work")
updates=[
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)     # insert
]
columns = ["id", "customer_id", "product", "amount"]
update_df=spark.createDataFrame(updates,columns)
update_df.display()

id,customer_id,product,amount
3,C003,Tablet,22000
6,C006,Watch,8000


In [0]:


deltatable.alias("t").merge(update_df.alias("u"),"t.id=u.id").whenMatchedUpdate(set={"amount":"u.amount"}
    ).whenNotMatchedInsert(values={
    'id':'u.id','customer_id':'u.customer_id','product':'u.product','amount':'u.amount'
}).execute()



deltatable.toDF().display()

id,customer_id,product,amount
2,C002,Mobile,18000
4,C004,Laptop,55000
5,C005,Camera,30000
3,C003,Tablet,22000
6,C006,Watch,8000


###TASK 6: Schema Evolution


In [0]:
# Add new column:

#country

data=[(7,'C007','Headphones',10000,'USA'),
      (8,'C008','Keyboard',20000,'Canada')]
new_df=spark.createDataFrame(data,['id','customer_id','product','amount','country'])
new_df.display()

id,customer_id,product,amount,country
7,C007,Headphones,10000,USA
8,C008,Keyboard,20000,Canada


In [0]:
new_df.write.format("delta").mode("append").option("mergeSchema","true").save("/Volumes/volume1/default/delta_work")
df_new=spark.read.load("/Volumes/volume1/default/delta_work")
df_new.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7080130323399825>, line 1
----> 1 new_df.write.format("delta").mode("append").option("mergeSchema","true").save("/Volumes/volume1/default/delta_work")
      2 df_new=spark.read.load("/Volumes/volume1/default/delta_work")
      3 df_new.display()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
 



###TASK 7:

In [0]:
#Previous version
#Restore old data

deltatable.history().display()

In [0]:


df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/volume1/default/delta_work")
df_old.display()

In [0]:

deltatable.restoreToVersion(0)

In [0]:
deltatable.toDF().display()